In [3]:
"""
Kalman xG → Win/Draw/Loss kansen via Dixon-Coles Poisson
=========================================================
Input:  Kalman data/kalman_expected_goals.csv
Output: Kalman data/kalman_probabilities.csv

Stappen:
1. Laad Kalman xG voorspellingen
2. Schaal naar werkelijk xG niveau (ratio 1.127)
3. Dixon-Coles Poisson correctie voor lage scores
4. Sommeer over scorelijnen → P(H), P(D), P(A)
"""

import numpy as np
import pandas as pd
from scipy.stats import poisson

DATA_PATH = r"C:/Users/semwi/FPL-Core-Insights/data/Kalman data/kalman_expected_goals.csv"
OUT_PATH  = r"C:/Users/semwi/FPL-Core-Insights/data/Kalman data/kalman_probabilities.csv"

# ── Parameters ────────────────────────────────────────────────────────────────

# Empirische scaling: Kalman underpredicts gemiddeld met factor 1.127
# Berekend als: werkelijk_xg_gemiddeld / kalman_xg_pred_gemiddeld
SCALE_HOME = 1.1265   # home: 1.6357 / 1.4519
SCALE_AWAY = 1.1264   # away: 1.3596 / 1.2071

# Dixon-Coles rho: correctiefactor voor lage scores
# Negatief = minder 0-0 dan standaard Poisson verwacht
RHO = -0.13

MAX_GOALS = 8   # Sommeer scorelijnen tot 8-8

# ── Dixon-Coles correctiefactor ───────────────────────────────────────────────

def dixon_coles_tau(i, j, lambda_h, lambda_a, rho=RHO):
    """
    Correctiefactor voor lage scores (Dixon & Coles, 1997).

    Standaard Poisson overschat de kans op 0-0, 1-0, 0-1, 1-1.
    Deze factor past die kansen aan op basis van rho:
      rho < 0: minder gelijke lage scores dan Poisson voorspelt
      rho = 0: identiek aan standaard Poisson
    """
    if   i == 0 and j == 0: return 1 - lambda_h * lambda_a * rho
    elif i == 1 and j == 0: return 1 + lambda_a * rho
    elif i == 0 and j == 1: return 1 + lambda_h * rho
    elif i == 1 and j == 1: return 1 - rho
    else:                   return 1.0


# ── Kansen berekenen ──────────────────────────────────────────────────────────

def xg_to_probabilities(kalman_xg_home, kalman_xg_away):
    """
    Converteer Kalman xG voorspellingen naar P(Home), P(Draw), P(Away).

    Stap 1: clip negatieve waarden (kunnen voorkomen bij zwakke teams)
    Stap 2: schaal naar werkelijk xG niveau
    Stap 3: Poisson PMF per schorlijn × Dixon-Coles correctie
    Stap 4: sommeer over alle scorelijnen

    Returns: (p_home, p_draw, p_away)
    """
    # Stap 1: clip
    lh = max(kalman_xg_home, 0.05)
    la = max(kalman_xg_away, 0.05)

    # Stap 2: schaal
    lh = lh * SCALE_HOME
    la = la * SCALE_AWAY

    # Stap 3 & 4: sommeer over scorelijnen
    p_home = p_draw = p_away = 0.0
    for i in range(MAX_GOALS + 1):
        for j in range(MAX_GOALS + 1):
            p_ij = (poisson.pmf(i, lh) *
                    poisson.pmf(j, la) *
                    dixon_coles_tau(i, j, lh, la))
            if   i > j: p_home += p_ij
            elif i == j: p_draw += p_ij
            else:        p_away += p_ij

    # Normaliseer (sommeer naar 1 door floating point)
    total = p_home + p_draw + p_away
    return round(p_home / total, 5), round(p_draw / total, 5), round(p_away / total, 5)


# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    print("=" * 60)
    print("  KALMAN xG → POISSON KANSEN (Dixon-Coles)")
    print("=" * 60)

    df = pd.read_csv(DATA_PATH, parse_dates=["date"])
    df = df.sort_values("date").reset_index(drop=True)

    print(f"\nLoaded {len(df)} wedstrijden")
    print(f"Periode: {df['date'].min().date()} → {df['date'].max().date()}")
    print(f"\nScaling: HOME × {SCALE_HOME}, AWAY × {SCALE_AWAY}")
    print(f"Dixon-Coles ρ = {RHO}")

    # Alleen wedstrijden met Kalman xG voorspelling
    valid = df[df['kalman_xg_pred_home'].notna()].copy()
    print(f"Wedstrijden met Kalman xG pred: {len(valid)}")

    # Bereken kansen
    print("\nKansen berekenen...")
    probs = valid.apply(
        lambda r: xg_to_probabilities(
            r['kalman_xg_pred_home'],
            r['kalman_xg_pred_away']
        ),
        axis=1
    )

    valid['kalman_prob_home'] = [p[0] for p in probs]
    valid['kalman_prob_draw'] = [p[1] for p in probs]
    valid['kalman_prob_away'] = [p[2] for p in probs]

    # Scaled lambda's ook opslaan voor transparantie
    valid['lambda_home'] = valid['kalman_xg_pred_home'].clip(lower=0.05) * SCALE_HOME
    valid['lambda_away'] = valid['kalman_xg_pred_away'].clip(lower=0.05) * SCALE_AWAY

    # ── Statistieken ──────────────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("  STATISTIEKEN KALMAN KANSEN")
    print("=" * 60)

    print(f"\nGemiddelde kansen:")
    print(f"  P(Home win) : {valid['kalman_prob_home'].mean():.3f}")
    print(f"  P(Draw)     : {valid['kalman_prob_draw'].mean():.3f}")
    print(f"  P(Away win) : {valid['kalman_prob_away'].mean():.3f}")
    print(f"  Som check   : {(valid['kalman_prob_home'] + valid['kalman_prob_draw'] + valid['kalman_prob_away']).mean():.6f}")

    print(f"\nBereik P(Home win):")
    print(f"  Min: {valid['kalman_prob_home'].min():.3f}")
    print(f"  Max: {valid['kalman_prob_home'].max():.3f}")
    print(f"  Std: {valid['kalman_prob_home'].std():.3f}")

    print(f"\nBereik P(Draw):")
    print(f"  Min: {valid['kalman_prob_draw'].min():.3f}")
    print(f"  Max: {valid['kalman_prob_draw'].max():.3f}")
    print(f"  Std: {valid['kalman_prob_draw'].std():.3f}")

    print(f"\nBereik P(Away win):")
    print(f"  Min: {valid['kalman_prob_away'].min():.3f}")
    print(f"  Max: {valid['kalman_prob_away'].max():.3f}")
    print(f"  Std: {valid['kalman_prob_away'].std():.3f}")

    # Meest extreme voorspellingen
    print(f"\nTop 5 grootste thuiswin kansen:")
    top_home = valid.nlargest(5, 'kalman_prob_home')[
        ['date','home_team','away_team',
         'kalman_xg_pred_home','kalman_xg_pred_away',
         'lambda_home','lambda_away',
         'kalman_prob_home','kalman_prob_draw','kalman_prob_away']
    ]
    print(top_home.to_string(index=False))

    print(f"\nTop 5 grootste uitwin kansen:")
    top_away = valid.nlargest(5, 'kalman_prob_away')[
        ['date','home_team','away_team',
         'kalman_xg_pred_home','kalman_xg_pred_away',
         'lambda_home','lambda_away',
         'kalman_prob_home','kalman_prob_draw','kalman_prob_away']
    ]
    print(top_away.to_string(index=False))

    print(f"\nTop 5 grootste gelijkspel kansen:")
    top_draw = valid.nlargest(5, 'kalman_prob_draw')[
        ['date','home_team','away_team',
         'kalman_xg_pred_home','kalman_xg_pred_away',
         'lambda_home','lambda_away',
         'kalman_prob_home','kalman_prob_draw','kalman_prob_away']
    ]
    print(top_draw.to_string(index=False))

    # Per seizoen
    print(f"\nGemiddelde kansen per seizoen:")
    print(f"{'Seizoen':<12} {'P(Home)':>8} {'P(Draw)':>8} {'P(Away)':>8}")
    print("-" * 40)
    for season in sorted(valid['season'].unique()):
        s = valid[valid['season'] == season]
        print(f"{season:<12} {s['kalman_prob_home'].mean():>8.3f} "
              f"{s['kalman_prob_draw'].mean():>8.3f} "
              f"{s['kalman_prob_away'].mean():>8.3f}")

    # ── Opslaan ───────────────────────────────────────────────────────────────
    output_cols = [
        'date', 'season', 'home_team', 'away_team',
        'home_goals', 'away_goals',
        'kalman_xg_pred_home', 'kalman_xg_pred_away',
        'lambda_home', 'lambda_away',
        'kalman_prob_home', 'kalman_prob_draw', 'kalman_prob_away',
        'home_pi_rating_pre', 'away_pi_rating_pre',
        'home_lineup_strength', 'away_lineup_strength',
        'kalman_alpha_home', 'kalman_gamma_home',
        'kalman_alpha_away', 'kalman_gamma_away',
    ]
    # Alleen kolommen die bestaan
    output_cols = [c for c in output_cols if c in valid.columns]
    out = valid[output_cols].copy()
    out.to_csv(OUT_PATH, index=False)

    print(f"\nOpgeslagen: {OUT_PATH}")
    print(f"  {len(out)} rijen, {len(output_cols)} kolommen")
    print(f"\nNieuwe kolommen:")
    print(f"  lambda_home        : geschaalde xG thuisploeg (Poisson parameter)")
    print(f"  lambda_away        : geschaalde xG uitploeg (Poisson parameter)")
    print(f"  kalman_prob_home   : P(thuisploeg wint)")
    print(f"  kalman_prob_draw   : P(gelijkspel)")
    print(f"  kalman_prob_away   : P(uitploeg wint)")


if __name__ == "__main__":
    main()

  KALMAN xG → POISSON KANSEN (Dixon-Coles)

Loaded 2608 wedstrijden
Periode: 2019-08-09 → 2026-03-22

Scaling: HOME × 1.1265, AWAY × 1.1264
Dixon-Coles ρ = -0.13
Wedstrijden met Kalman xG pred: 2608

Kansen berekenen...

  STATISTIEKEN KALMAN KANSEN

Gemiddelde kansen:
  P(Home win) : 0.435
  P(Draw)     : 0.244
  P(Away win) : 0.321
  Som check   : 1.000000

Bereik P(Home win):
  Min: 0.008
  Max: 0.986
  Std: 0.176

Bereik P(Draw):
  Min: 0.012
  Max: 0.342
  Std: 0.043

Bereik P(Away win):
  Min: 0.002
  Max: 0.957
  Std: 0.160

Top 5 grootste thuiswin kansen:
      date       home_team            away_team  kalman_xg_pred_home  kalman_xg_pred_away  lambda_home  lambda_away  kalman_prob_home  kalman_prob_draw  kalman_prob_away
2020-09-19         Everton West Bromwich Albion               5.2243               0.3447     5.885174     0.388270           0.98578           0.01222           0.00200
2020-07-26 Manchester City         Norwich City               3.7508               0.4505 